In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\mrabe\OneDrive\Documentos\Premier League Analysis\Data\matches_limpio.csv")
df['Date'] = pd.to_datetime(df['Date'])
print(f"Dataset cargado: {df.shape[0]} partidos ✓")

Dataset cargado: 12026 partidos ✓


In [8]:
resultados = df['FTR'].value_counts()
resultados.index = resultados.index.map({'H': 'Local gana', 'A': 'Visitante gana', 'D': 'Empate'})
print(resultados.to_string())
print(f"\nEl local gana el {round(resultados['Local gana'] / resultados.sum() * 100, 1)}% de los partidos")

FTR
Local gana        5519
Visitante gana    3410
Empate            3097

El local gana el 45.9% de los partidos


In [6]:
goles_temporada = df.groupby('Season_End_Year')['TotalGoals'].mean().round(2)
print("Promedio de goles por partido según temporada:")
print(goles_temporada.to_string())

Promedio de goles por partido según temporada:
Season_End_Year
1993    2.65
1994    2.59
1995    2.59
1996    2.60
1997    2.55
1998    2.68
1999    2.52
2000    2.79
2001    2.61
2002    2.63
2003    2.63
2004    2.66
2005    2.57
2006    2.48
2007    2.45
2008    2.64
2009    2.48
2010    2.77
2011    2.80
2012    2.81
2013    2.80
2014    2.77
2015    2.57
2016    2.70
2017    2.80
2018    2.68
2019    2.82
2020    2.72
2021    2.69
2022    2.82
2023    2.85


In [9]:
locales = df[df['FTR'] == 'H']['Home'].value_counts().head(10)
print("Top 10 equipos con más victorias de local:")
print(locales.to_string())

Top 10 equipos con más victorias de local:
Home
Manchester Utd     417
Arsenal            382
Liverpool          376
Chelsea            360
Tottenham          323
Manchester City    297
Everton            273
Newcastle Utd      264
West Ham           224
Aston Villa        220


In [10]:
goles_local = df.groupby('Home')['HomeGoals'].sum().sort_values(ascending=False).head(10)
print("Top 10 equipos con más goles de local:")
print(goles_local.to_string())

Top 10 equipos con más goles de local:
Home
Manchester Utd     1250
Arsenal            1207
Liverpool          1202
Chelsea            1141
Tottenham          1032
Manchester City    1017
Everton             897
Newcastle Utd       867
West Ham            756
Aston Villa         717


In [11]:
temp = df[df['Season_End_Year'] == 2023].copy()
print(f"Partidos en la temporada 2023: {len(temp)}")

Partidos en la temporada 2023: 380


In [12]:
# Puntos para el equipo local
def puntos_local(resultado):
    if resultado == 'H':   # ganó el local
        return 3
    elif resultado == 'D': # empate
        return 1
    else:                  # ganó el visitante
        return 0

def puntos_visitante(resultado):
    if resultado == 'A':   # ganó el visitante
        return 3
    elif resultado == 'D': # empate
        return 1
    else:                  # ganó el local
        return 0

temp['PuntosLocal'] = temp['FTR'].apply(puntos_local)
temp['PuntosVisitante'] = temp['FTR'].apply(puntos_visitante)

print("Columnas de puntos creadas ✓")
temp.head()

Columnas de puntos creadas ✓


,Season_End_Year,Wk,Date,Home,HomeGoals,AwayGoals,Away,FTR,TotalGoals,PuntosLocal,PuntosVisitante
11646,2023,1,2022-08-05,Crystal Palace,0,2,Arsenal,A,2,0,3
11647,2023,1,2022-08-06,Fulham,2,2,Liverpool,D,4,1,1
11648,2023,1,2022-08-06,Tottenham,4,1,Southampton,H,5,3,0
11649,2023,1,2022-08-06,Newcastle Utd,2,0,Nott'ham Forest,H,2,3,0
11650,2023,1,2022-08-06,Leeds United,2,1,Wolves,H,3,3,0


In [13]:
# Puntos sumados como local
como_local = temp.groupby('Home').agg(
    PuntosLocal = ('PuntosLocal', 'sum'),
    GolesAFavor_Local = ('HomeGoals', 'sum'),
    GolesEnContra_Local = ('AwayGoals', 'sum')
).reset_index().rename(columns={'Home': 'Equipo'})

# Puntos sumados como visitante
como_visitante = temp.groupby('Away').agg(
    PuntosVisitante = ('PuntosVisitante', 'sum'),
    GolesAFavor_Visitante = ('AwayGoals', 'sum'),
    GolesEnContra_Visitante = ('HomeGoals', 'sum')
).reset_index().rename(columns={'Away': 'Equipo'})

# Unir todo
tabla = como_local.merge(como_visitante, on='Equipo')

# Calcular totales
tabla['Puntos'] = tabla['PuntosLocal'] + tabla['PuntosVisitante']
tabla['GolesAFavor'] = tabla['GolesAFavor_Local'] + tabla['GolesAFavor_Visitante']
tabla['GolesEnContra'] = tabla['GolesEnContra_Local'] + tabla['GolesEnContra_Visitante']
tabla['DiferenciaGoles'] = tabla['GolesAFavor'] - tabla['GolesEnContra']

# Ordenar por puntos
tabla = tabla[['Equipo', 'Puntos', 'GolesAFavor', 'GolesEnContra', 'DiferenciaGoles']]
tabla = tabla.sort_values('Puntos', ascending=False).reset_index(drop=True)
tabla.index += 1  # Posición empieza en 1

print(tabla)

             Equipo  Puntos  GolesAFavor  GolesEnContra  DiferenciaGoles
1   Manchester City      89           94             33               61
2           Arsenal      84           88             43               45
3    Manchester Utd      75           58             43               15
4     Newcastle Utd      71           68             33               35
5         Liverpool      67           75             47               28
6          Brighton      62           72             53               19
7       Aston Villa      61           51             46                5
8         Tottenham      60           70             63                7
9         Brentford      59           58             46               12
10           Fulham      52           55             53                2
11   Crystal Palace      45           40             49               -9
12          Chelsea      44           38             47               -9
13           Wolves      41           31           

In [14]:
tabla.to_csv(r"C:\Users\mrabe\OneDrive\Documentos\Premier League Analysis\Data\tabla_posiciones_2023.csv")
print("Tabla de posiciones guardada ✓")

Tabla de posiciones guardada ✓
